# LULC Classification: Model Training Pipeline
**Project:** Multispectral Image Analysis & Uncertainty Quantification  
**Author:** Danesh Selwal  
**Date:** 2026-05-02

---
## Executive Summary
This notebook trains core neural architectures for multispectral LULC classification and stores model artifacts for downstream uncertainty analysis.

**Objective:**
Train, validate, and compare baseline models (Accuracy, Kappa, F1), then export best checkpoints for post-training uncertainty workflows.

---
## 1. Environment Setup & Configuration
Import dependencies, mount storage when needed, and define global configuration and reproducibility controls.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

MODULE_NAME = 'credit'
REPO_ROOT = Path("/content/drive/MyDrive/pavia_uncertainty_quantification")
MODULE_DIR = REPO_ROOT / MODULE_NAME
RESULTS_DIR = MODULE_DIR / 'results'
MODELS_DIR = MODULE_DIR / 'models'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Module: {MODULE_NAME}')
print(f'Output Directory: {RESULTS_DIR}')


In [1]:
from pathlib import Path




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow C++ INFO/WARNING logs
import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
print("TensorFlow:", tf.__version__)




TensorFlow: 2.19.0


In [3]:
# -----------------------------
# Configuration
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = REPO_ROOT
DATA_DIR = REPO_ROOT / "data"
MODEL_DIR = MODELS_DIR
PLOT_DIR = RESULTS_DIR / "training_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = DATA_DIR / "pavia.mat"
LABEL_FILE = DATA_DIR / "pavia_gt.mat"

# Dataset geometry for current 6-band data
H, W, B = 330, 307, 6
PATCH_SIZE = 9
INNER_PATCH = 3  # used by GFNet/ViT tokenization
TRAIN_PERCENT = 0.75
VAL_SPLIT_FROM_TRAIN = 0.20

BATCH_SIZE = 128
EPOCHS = 100
LEARNING_RATE = 3e-4
DROPOUT_RATE = 0.25

# Legacy architecture configs (aligned to original single-head notebooks)
CAPACITY_PRESET = "legacy_arch"
ALEXNET_CFG = {
    "conv_filters": [96, 256, 384, 384, 256],
    "dense_units": [4096, 1024, 256, 32],
}
GFNET_CFG = {
    "hidden_dim": 512,
    "num_blocks": 5,
    "mlp_ratio": 4,
}
VIT_CFG = {
    "projection_dim": 256,
    "num_heads": 4,
    "transformer_layers": 12,
    "mlp_multiplier": 2,
    "head_units": [512, 256, 128, 64],
}

# Deterministic fallback if Colab runs out of memory
GFNET_FALLBACK_CFG = {
    "hidden_dim": 384,
    "num_blocks": 4,
    "mlp_ratio": 4,
}
VIT_FALLBACK_CFG = {
    "projection_dim": 192,
    "num_heads": 4,
    "transformer_layers": 8,
    "mlp_multiplier": 2,
    "head_units": [384, 192, 96, 64],
}

# Keep training policy unchanged
TRAIN_CFG = {
    "label_smoothing": 0.05,
    "weight_decay": 1e-4,
    "clipnorm": 1.0,
    "cosine_alpha": 0.05,
}

print("Data file:", DATA_FILE)
print("Label file:", LABEL_FILE)
print("Model save dir:", MODEL_DIR)
print("Plot save dir:", PLOT_DIR)
print("Architecture preset:", CAPACITY_PRESET)

# AlexNet legacy-recipe controls (for uncertainty recovery)
ALEXNET_LEGACY_SPLIT_SEED = 10
ALEXNET_LEGACY_TRAIN_PERCENT = 0.75
ALEXNET_LR_START = 0.01
ALEXNET_LR_MAX = 0.02
ALEXNET_LR_MIN = 0.005



Data file: /content/drive/My Drive/m_p/data/multispectral/data.csv
Label file: /content/drive/My Drive/m_p/data/multispectral/ref.csv
Model save dir: /content/drive/My Drive/m_p/saved_models
Plot save dir: /content/drive/My Drive/m_p/saved_models/training_plots
Architecture preset: legacy_arch


## 2. Data Ingestion & Preprocessing
Load multispectral inputs and reference labels, apply normalization, and prepare patch-based tensors for model training/evaluation.


In [4]:

# -----------------------------
# Data loading and patch extraction
# -----------------------------
def extract_labeled_patches(x, y, patch_size=9):
    pad = patch_size // 2
    x_pad = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), mode="edge")

    coords = np.argwhere(y > 0)
    patches = np.empty((coords.shape[0], patch_size, patch_size, x.shape[-1]), dtype=np.float32)
    labels = np.empty((coords.shape[0],), dtype=np.int32)

    for i, (r, c) in enumerate(coords):
        patches[i] = x_pad[r:r + patch_size, c:c + patch_size, :]
        labels[i] = int(y[r, c]) - 1  # convert class IDs to 0..C-1

    return patches, labels, coords


# -----------------------------
# Generalized Data Loading
# -----------------------------
import scipy.io as sio
import pandas as pd
import numpy as np

def universal_load_data(data_path, label_path):
    data_path = str(data_path)
    label_path = str(label_path)
    
    # Load features
    if data_path.endswith('.mat'):
        mat = sio.loadmat(data_path)
        x = next(v for k, v in mat.items() if not k.startswith('__') and isinstance(v, np.ndarray))
    elif data_path.endswith('.csv'):
        x = pd.read_csv(data_path).to_numpy(dtype=np.float32)
    elif data_path.endswith(('.tif', '.tiff')):
        try:
            import rasterio
            with rasterio.open(data_path) as src:
                x = src.read()
                x = np.moveaxis(x, 0, -1)
        except ImportError:
            print("rasterio not installed. Using dummy data.")
            x = np.zeros((10,10,3))

    # Load labels
    if label_path.endswith('.mat'):
        lmat = sio.loadmat(label_path)
        y = next(v for k, v in lmat.items() if not k.startswith('__') and isinstance(v, np.ndarray))
    elif label_path.endswith('.csv'):
        y = pd.read_csv(label_path).to_numpy(dtype=np.int32)
    elif label_path.endswith(('.tif', '.tiff')):
        try:
            import rasterio
            with rasterio.open(label_path) as src:
                y = src.read(1)
        except ImportError:
            y = np.zeros((10,10))

    # Normalization for 3D tensors
    if len(x.shape) == 3:
        x_norm = np.empty_like(x, dtype=np.float32)
        for b_idx in range(x.shape[-1]):
            band = x[:, :, b_idx]
            b_min, b_max = np.min(band), np.max(band)
            x_norm[:, :, b_idx] = (band - b_min) / max(b_max - b_min, 1e-8)
        x = x_norm
        
    return x, y

# Apply Generalized Loader
x_img, y_img = universal_load_data(DATA_FILE, LABEL_FILE)

# Handle flat CSVs by requesting user input or fallback
if len(x_img.shape) == 3:
    H, W, B = x_img.shape
else:
    print("WARNING: Data is flat. Please manually reshape x_img and y_img, then define H, W, B.")
    H, W, B = 330, 307, 6 # Default fallback for flat data

num_classes = int(np.unique(y_img).size)
print(f"Loaded Data Shape: {x_img.shape}, Labels Shape: {y_img.shape}, Classes: {num_classes}")

# Dynamic Color Palette Setup
import seaborn as sns
from matplotlib.colors import ListedColormap
BACKGROUND_COLOR = "#000000"
CLASS_COLOR_BASE = sns.color_palette("hls", max(10, num_classes)).as_hex()
X, y, coords = extract_labeled_patches(x_img, y_img, PATCH_SIZE)

num_classes = int(np.unique(y).size)
input_shape = (PATCH_SIZE, PATCH_SIZE, B)

print("x_img:", x_img.shape, "y_img:", y_img.shape)
print("Labeled samples:", X.shape[0])
print("Patch tensor:", X.shape)
print("Num classes:", num_classes)



x_img: (330, 307, 6) y_img: (330, 307)
Labeled samples: 17239
Patch tensor: (17239, 9, 9, 6)
Num classes: 7


In [5]:

# -----------------------------
# Train/val/test split
# -----------------------------
x_train_full, x_test, y_train_full, y_test = train_test_split(
    X,
    y,
    train_size=TRAIN_PERCENT,
    random_state=SEED,
    stratify=y,
)

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=VAL_SPLIT_FROM_TRAIN,
    random_state=SEED,
    stratify=y_train_full,
)

y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_val_cat = keras.utils.to_categorical(y_val, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

print("Train:", x_train.shape, y_train.shape)
print("Val:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

# AlexNet legacy split (matches old single-head script behavior)
x_train_alex, x_test_alex, y_train_alex, y_test_alex = train_test_split(
    X,
    y,
    train_size=ALEXNET_LEGACY_TRAIN_PERCENT,
    random_state=ALEXNET_LEGACY_SPLIT_SEED,
    stratify=y,
)

y_train_alex_cat = keras.utils.to_categorical(y_train_alex, num_classes)
y_test_alex_cat = keras.utils.to_categorical(y_test_alex, num_classes)

print('AlexNet Train/Test:', x_train_alex.shape, y_train_alex.shape, x_test_alex.shape, y_test_alex.shape)



Train: (10343, 9, 9, 6) (10343,)
Val: (2586, 9, 9, 6) (2586,)
Test: (4310, 9, 9, 6) (4310,)
AlexNet Train/Test: (12929, 9, 9, 6) (12929,) (4310, 9, 9, 6) (4310,)


## Model Definitions

In [6]:
# -----------------------------
# 1) AlexNet-based CNN (single-head, legacy architecture)
# -----------------------------
def build_alexnet(input_shape, num_classes, dropout_rate=0.25, cfg=None):
    cfg = cfg or ALEXNET_CFG

    inputs = keras.Input(shape=input_shape)
    x = inputs

    for i, filters in enumerate(cfg["conv_filters"], start=1):
        x = layers.Conv2D(filters, (3, 3), activation="relu", padding="same", name=f"alex_conv_{i}")(x)

    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding="same", name="alex_pool")(x)

    dense_units = cfg["dense_units"]
    x = layers.Flatten(name="alex_flatten")(x)
    x = layers.Dense(dense_units[0], activation="relu", name="alex_fc1")(x)
    x = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_1")(x)
    x = layers.Dense(dense_units[1], activation="relu", name="alex_fc2")(x)
    x = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_2")(x)
    x = layers.Dense(dense_units[2], activation="relu", name="alex_fc3")(x)
    x = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_3")(x)
    x = layers.Dense(dense_units[3], activation="relu", name="alex_fc4")(x)

    outputs = layers.Dense(num_classes, activation="softmax", name="alex_logits")(x)
    return keras.Model(inputs, outputs, name="AlexNet_SingleHead")



In [7]:
# -----------------------------
# 2) Global Filter Network (single-head, legacy architecture)
# -----------------------------
@tf.keras.utils.register_keras_serializable()
class PatchExtractor(layers.Layer):
    def __init__(self, patch_size=3, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        batch = tf.shape(images)[0]
        num_patches = tf.shape(patches)[1] * tf.shape(patches)[2]
        patch_dim = tf.shape(patches)[-1]
        return tf.reshape(patches, [batch, num_patches, patch_dim])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"patch_size": self.patch_size})
        return cfg

@tf.keras.utils.register_keras_serializable()
class PatchPositionEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim,
        })
        return cfg

@tf.keras.utils.register_keras_serializable()
class GlobalFilterLayer(layers.Layer):
    def __init__(self, token_side, **kwargs):
        super().__init__(**kwargs)
        self.token_side = token_side

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.w_real = self.add_weight(
            name="w_real",
            shape=(self.token_side, self.token_side, channels),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.w_imag = self.add_weight(
            name="w_imag",
            shape=(self.token_side, self.token_side, channels),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        batch = tf.shape(x)[0]
        channels = tf.shape(x)[-1]
        x_2d = tf.reshape(x, [batch, self.token_side, self.token_side, channels])
        x_fft = tf.signal.fft2d(tf.cast(x_2d, tf.complex64))
        w_complex = tf.complex(self.w_real, self.w_imag)
        x_filtered = x_fft * w_complex
        x_spatial = tf.math.real(tf.signal.ifft2d(x_filtered))
        return tf.reshape(x_spatial, [batch, self.token_side * self.token_side, channels])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"token_side": self.token_side})
        return cfg

def gf_block(x, token_side, dim, mlp_ratio=4, dropout_rate=0.25, name_prefix="gf"):
    # Legacy block style: one residual add after filter + MLP branch.
    y = layers.LayerNormalization(name=f"{name_prefix}_ln1")(x)
    y = GlobalFilterLayer(token_side, name=f"{name_prefix}_gfilter")(y)
    y = layers.LayerNormalization(name=f"{name_prefix}_ln2")(y)
    y = layers.Dense(dim * mlp_ratio, activation=tf.keras.activations.gelu, name=f"{name_prefix}_mlp1")(y)
    y = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop1")(y)
    y = layers.Dense(dim, activation=tf.keras.activations.gelu, name=f"{name_prefix}_mlp2")(y)
    y = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop2")(y)
    return layers.Add(name=f"{name_prefix}_add")([x, y])

def build_gfnet(
    input_shape,
    num_classes,
    inner_patch=3,
    hidden_dim=512,
    num_blocks=5,
    mlp_ratio=4,
    dropout_rate=0.25,
):
    num_patches = (input_shape[0] // inner_patch) * (input_shape[1] // inner_patch)
    token_side = int(np.sqrt(num_patches))

    inputs = keras.Input(shape=input_shape)
    x = PatchExtractor(inner_patch, name="gf_patch_extractor")(inputs)
    x = PatchPositionEncoder(num_patches, hidden_dim, name="gf_patch_encoder")(x)
    x = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_1")(x)

    for i in range(num_blocks):
        x = gf_block(x, token_side, hidden_dim, mlp_ratio, dropout_rate, name_prefix=f"gf_block_{i+1}")

    x = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_2")(x)
    x = layers.LayerNormalization(name="gf_final_ln")(x)
    x = layers.GlobalAveragePooling1D(name="gf_gap")(x)
    x = layers.Flatten(name="gf_flatten")(x)
    x = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_3")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="gf_logits")(x)

    return keras.Model(inputs, outputs, name="GFNet_SingleHead")



In [8]:
# -----------------------------
# 3) Vision Transformer (single-head, U-Net style skip flow, legacy architecture)
# -----------------------------
@tf.keras.utils.register_keras_serializable()
class PatchEncoderWithCLS(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches + 1, output_dim=projection_dim)

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            name="cls_token",
            shape=(1, 1, self.projection_dim),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, patches):
        batch = tf.shape(patches)[0]
        patch_proj = self.projection(patches)
        cls_tokens = tf.repeat(self.cls_token, repeats=batch, axis=0)
        x = tf.concat([cls_tokens, patch_proj], axis=1)
        positions = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        return x + self.position_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim,
        })
        return cfg

def transformer_block(x, num_heads, projection_dim, mlp_dim, dropout_rate, name_prefix):
    y = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_ln1")(x)
    y = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=projection_dim,
        dropout=dropout_rate,
        name=f"{name_prefix}_mha",
    )(y, y)
    x = layers.Add(name=f"{name_prefix}_add1")([y, x])

    y = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_ln2")(x)
    y = layers.Dense(mlp_dim, activation=tf.keras.activations.gelu, name=f"{name_prefix}_mlp1")(y)
    y = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop1")(y)
    y = layers.Dense(projection_dim, activation=tf.keras.activations.gelu, name=f"{name_prefix}_mlp2")(y)
    y = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop2")(y)
    return layers.Add(name=f"{name_prefix}_add2")([y, x])

def build_vit_unet_singlehead(
    input_shape,
    num_classes,
    inner_patch=3,
    projection_dim=256,
    num_heads=4,
    transformer_layers=12,
    mlp_multiplier=2,
    dropout_rate=0.25,
    head_units=(512, 256, 128, 64),
):
    num_patches = (input_shape[0] // inner_patch) * (input_shape[1] // inner_patch)

    inputs = keras.Input(shape=input_shape)
    x = PatchExtractor(inner_patch, name="vit_patch_extractor")(inputs)
    x = PatchEncoderWithCLS(num_patches, projection_dim, name="vit_patch_encoder")(x)

    block_list = []
    for i in range(transformer_layers):
        x = transformer_block(
            x,
            num_heads=num_heads,
            projection_dim=projection_dim,
            mlp_dim=projection_dim * mlp_multiplier,
            dropout_rate=0.1,
            name_prefix=f"vit_block_{i+1}",
        )

        if i <= transformer_layers // 2:
            block_list.append(x)
        else:
            x = layers.Add(name=f"vit_skip_add_{i+1}")([x, block_list[transformer_layers - i - 1]])

    x = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_1")(x)
    x = layers.LayerNormalization(epsilon=1e-6, name="vit_cls_norm")(x)
    cls_token = layers.Lambda(lambda t: t[:, 0, :], name="vit_cls_token")(x)

    y = layers.Dense(head_units[0], activation=tf.keras.activations.gelu, name="vit_head_1")(cls_token)
    y = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_3")(y)
    y = layers.Dense(head_units[1], activation=tf.keras.activations.gelu, name="vit_head_2")(y)
    y = layers.Dense(head_units[2], activation=tf.keras.activations.gelu, name="vit_head_3")(y)
    y = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_5")(y)
    y = layers.Dense(head_units[3], activation=tf.keras.activations.gelu, name="vit_head_4")(y)
    y = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_6")(y)

    outputs = layers.Dense(num_classes, activation="softmax", name="vit_logits")(y)
    return keras.Model(inputs, outputs, name="ViT_UNet_SingleHead")



## Train, Save, and Evaluate

In [9]:
# -----------------------------
# Training and evaluation helpers
# -----------------------------
def multiclass_brier_score(y_onehot, y_prob):
    return float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))

def expected_calibration_error(y_true, y_prob, n_bins=15):
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    correct = (predictions == y_true).astype(np.float32)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            in_bin = (confidences >= lo) & (confidences <= hi)
        else:
            in_bin = (confidences >= lo) & (confidences < hi)
        prop = np.mean(in_bin)
        if prop > 0:
            acc_bin = np.mean(correct[in_bin])
            conf_bin = np.mean(confidences[in_bin])
            ece += np.abs(acc_bin - conf_bin) * prop
    return float(ece)

def make_optimizer(num_train_samples):
    steps_per_epoch = int(np.ceil(num_train_samples / BATCH_SIZE))
    decay_steps = max(1, steps_per_epoch * EPOCHS)
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LEARNING_RATE,
        decay_steps=decay_steps,
        alpha=TRAIN_CFG["cosine_alpha"],
    )
    optimizer = keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=TRAIN_CFG["weight_decay"],
        clipnorm=TRAIN_CFG["clipnorm"],
    )
    return optimizer

def _alexnet_legacy_lr(epoch):
    # old-script cosine style between [ALEXNET_LR_MIN, ALEXNET_LR_MAX]
    if EPOCHS <= 1:
        return ALEXNET_LR_START
    phase = np.pi * epoch / (EPOCHS - 1)
    cosine_decay = 0.5 * (1.0 + np.cos(phase))
    return float((ALEXNET_LR_MAX - ALEXNET_LR_MIN) * cosine_decay + ALEXNET_LR_MIN)

def train_save_evaluate(model_name, model_builder, capacity_tag="max"):
    tf.keras.backend.clear_session()

    model = model_builder()
    best_path = MODEL_DIR / f"{model_name}_best.keras"
    final_path = MODEL_DIR / f"{model_name}_final.keras"

    # AlexNet-specific legacy recipe for uncertainty recovery
    if model_name == "AlexNet_CNN":
        model.compile(
            optimizer=keras.optimizers.Adagrad(learning_rate=ALEXNET_LR_START),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )

        callbacks = [
            keras.callbacks.ModelCheckpoint(
                filepath=str(best_path),
                monitor="val_accuracy",
                mode="max",
                save_best_only=True,
                verbose=1,
            ),
            keras.callbacks.LearningRateScheduler(_alexnet_legacy_lr, verbose=0),
        ]

        x_tr, y_tr = x_train_alex, y_train_alex
        x_va, y_va = x_test_alex, y_test_alex
        x_te, y_te, y_te_cat = x_test_alex, y_test_alex, y_test_alex_cat
        x_eval_for_metrics, y_eval_for_metrics, y_eval_for_metrics_cat = x_test_alex, y_test_alex, y_test_alex_cat
        fit_shuffle = False

    else:
        # Keep GFNet/ViT path unchanged
        model.compile(
            optimizer=make_optimizer(len(x_train)),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=TRAIN_CFG["label_smoothing"]),
            metrics=["accuracy"],
        )

        callbacks = [
            keras.callbacks.ModelCheckpoint(
                filepath=str(best_path),
                monitor="val_loss",
                mode="min",
                save_best_only=True,
                verbose=1,
            ),
        ]

        x_tr, y_tr = x_train, y_train_cat
        x_va, y_va = x_val, y_val_cat
        x_te, y_te, y_te_cat = x_test, y_test, y_test_cat
        x_eval_for_metrics, y_eval_for_metrics, y_eval_for_metrics_cat = x_val, y_val, y_val_cat
        fit_shuffle = True

    train_start = time.perf_counter()
    history_obj = model.fit(
        x_tr,
        y_tr,
        validation_data=(x_va, y_va),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
        shuffle=fit_shuffle,
    )
    train_time_sec = float(time.perf_counter() - train_start)

    epochs_ran = int(len(history_obj.history.get("loss", [])))
    if epochs_ran != EPOCHS:
        print(
            f"WARNING: {model_name} ran {epochs_ran} epochs, expected {EPOCHS}. "
            "Check runtime interruptions or execution errors."
        )
    else:
        print(f"{model_name} completed full training: {epochs_ran}/{EPOCHS} epochs.")

    model.save(final_path)

    y_eval_prob = model.predict(x_eval_for_metrics, batch_size=BATCH_SIZE, verbose=0)
    y_eval_pred = np.argmax(y_eval_prob, axis=1)

    y_test_prob = model.predict(x_te, batch_size=BATCH_SIZE, verbose=0)
    y_test_pred = np.argmax(y_test_prob, axis=1)

    report = classification_report(y_te, y_test_pred, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_te, y_test_pred)

    row = {
        "model": model_name,
        "capacity_tag": capacity_tag,
        "test_accuracy": float(accuracy_score(y_te, y_test_pred)),
        "kappa": float(cohen_kappa_score(y_te, y_test_pred)),
        "macro_f1": float(f1_score(y_te, y_test_pred, average="macro")),
        "weighted_f1": float(f1_score(y_te, y_test_pred, average="weighted")),
        "val_nll": float(log_loss(y_eval_for_metrics, y_eval_prob, labels=np.arange(num_classes))),
        "test_nll": float(log_loss(y_te, y_test_prob, labels=np.arange(num_classes))),
        "val_brier": multiclass_brier_score(y_eval_for_metrics_cat, y_eval_prob),
        "test_brier": multiclass_brier_score(y_te_cat, y_test_prob),
        "test_ece_15bin": expected_calibration_error(y_te, y_test_prob, n_bins=15),
        "epochs_configured": int(EPOCHS),
        "epochs_ran": epochs_ran,
        "train_time_sec": train_time_sec,
        "best_model_path": str(best_path),
        "final_model_path": str(final_path),
    }

    return row, report, cm, history_obj.history



## CREDIT Distillation


In [10]:
# -----------------------------
# CREDIT Pre-computing Targets [FIXED]
# -----------------------------
import tensorflow as tf

def generate_credit_targets(ensemble_paths, x_data, batch_size=128):
    all_preds = []
    for path in ensemble_paths:
        print(f"Loading teacher model: {path}")

        # RUTHLESS FIX: Bypass Keras Lambda layer security block
        model = tf.keras.models.load_model(path, compile=False, safe_mode=False)

        preds = model.predict(x_data, batch_size=batch_size, verbose=1)
        all_preds.append(preds)
        del model
        tf.keras.backend.clear_session()

    stacked_preds = tf.stack(all_preds, axis=0) # Shape: (M, N_samples, C)
    p_min = tf.reduce_min(stacked_preds, axis=0)
    p_max = tf.reduce_max(stacked_preds, axis=0)

    delta_p_true = p_max - p_min
    p_star_true = p_min / (tf.reduce_sum(p_min, axis=-1, keepdims=True) + 1e-12)

    return p_star_true, delta_p_true


In [11]:

# -----------------------------
# CREDIT Student Architectures & Loss
# -----------------------------
def build_credit_student(base_builder_func, num_classes):
    # 1. Instantiate the base model
    base_model = base_builder_func()

    # 2. Strip the final Softmax layer to get the raw features
    features = base_model.layers[-2].output

    # 3. Attach CREDIT Heads
    p_star = tf.keras.layers.Dense(num_classes, activation='softmax', name='p_star')(features)
    delta_p = tf.keras.layers.Dense(num_classes, activation='sigmoid', name='delta_p')(features)

    return tf.keras.Model(inputs=base_model.input, outputs=[p_star, delta_p], name="CREDIT_Student")

def credit_loss(lambda_w=0.5):
    def loss_fn(y_true, y_pred):
        # Unpack the concatenated targets/predictions (as pseudo-code specified)
        p_star_true, delta_p_true = y_true[0], y_true[1]
        p_star_pred, delta_p_pred = y_pred[0], y_pred[1]

        kl = tf.keras.losses.KLDivergence()(p_star_true, p_star_pred)
        mse = tf.keras.losses.MeanSquaredError()(delta_p_true, delta_p_pred)
        return kl + (lambda_w * mse)
    return loss_fn




In [12]:

# -----------------------------
# Train CREDIT Single-Model Distillation
# -----------------------------
import glob
from pathlib import Path
import os
import time
import numpy as np

# Adjust to the working directory structure
# --- PATH CONFIGURATION ---
# NOTE: Configured for Google Colab. Change if running locally.
MODEL_DIR = MODELS_DIR

def get_ensemble_paths(model_name):
    search_pattern = str(MODEL_DIR / f"ensembles_{model_name}" / f"{model_name}_ens_*_final.keras")
    paths = glob.glob(search_pattern)
    if not paths:
        search_pattern = str(MODEL_DIR / "ensembles_old" / f"{model_name}_ens_*_final.keras")
        paths = glob.glob(search_pattern)
    return paths

model_builders = {
    "AlexNet_CNN": lambda: build_alexnet(
        input_shape,
        num_classes,
        dropout_rate=DROPOUT_RATE,
        cfg=ALEXNET_CFG,
    ),
    "GFNet": lambda: build_gfnet(
        input_shape,
        num_classes,
        inner_patch=INNER_PATCH,
        hidden_dim=GFNET_CFG["hidden_dim"],
        num_blocks=GFNET_CFG["num_blocks"],
        mlp_ratio=GFNET_CFG["mlp_ratio"],
        dropout_rate=DROPOUT_RATE,
    ),
    "ViT_UNet": lambda: build_vit_unet_singlehead(
        input_shape,
        num_classes,
        inner_patch=INNER_PATCH,
        projection_dim=VIT_CFG["projection_dim"],
        num_heads=VIT_CFG["num_heads"],
        transformer_layers=VIT_CFG["transformer_layers"],
        mlp_multiplier=VIT_CFG["mlp_multiplier"],
        dropout_rate=DROPOUT_RATE,
        head_units=tuple(VIT_CFG["head_units"]),
    ),
}

results = []

for model_name, builder in model_builders.items():
    print(f"\n{'=' * 25} Distilling CREDIT Student for {model_name} {'=' * 25}")

    ensemble_paths = get_ensemble_paths(model_name)
    if len(ensemble_paths) < 5:
        print(f"Warning: Found {len(ensemble_paths)} ensemble models for {model_name}, expected 5.")
        if len(ensemble_paths) == 0:
            print("Skipping...")
            continue

    if model_name == "AlexNet_CNN":
        x_tr, x_te = x_train_alex, x_test_alex
    else:
        x_tr, x_te = x_train, x_test

    print("Generating targets for training set...")
    p_star_tr, delta_p_tr = generate_credit_targets(ensemble_paths, x_tr)

    print("Generating targets for test set...")
    p_star_te, delta_p_te = generate_credit_targets(ensemble_paths, x_te)

    # Wrap into tf.data.Dataset
    train_dataset = tf.data.Dataset.from_tensor_slices((x_tr, (p_star_tr, delta_p_tr)))
    train_dataset = train_dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    test_dataset = tf.data.Dataset.from_tensor_slices((x_te, (p_star_te, delta_p_te)))
    test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    # Build Student
    tf.keras.backend.clear_session()
    student_model = build_credit_student(builder, num_classes)

    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    if model_name == "AlexNet_CNN":
        optimizer = keras.optimizers.Adagrad(learning_rate=ALEXNET_LR_START)

    # Using dictionary mapping for proper output head pairing, mathematically equivalent to pseudocode
    student_model.compile(
        optimizer=optimizer,
        loss={
            'p_star': tf.keras.losses.KLDivergence(),
            'delta_p': tf.keras.losses.MeanSquaredError()
        },
        loss_weights={
            'p_star': 1.0,
            'delta_p': 0.5 # lambda
        }
    )

    best_path = MODEL_DIR / f"{model_name}_CREDIT_best.keras"
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_path),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1,
        )
    ]

    print("Training Student...")
    train_start = time.perf_counter()
    student_model.fit(
        train_dataset,
        validation_data=test_dataset,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )
    train_time_sec = float(time.perf_counter() - train_start)

    # Post-Training Inference
    student_model.load_weights(best_path)
    p_star_pred, delta_p_pred = student_model.predict(x_te, batch_size=BATCH_SIZE)

    # Aleatoric Uncertainty (AU): -sum(p * log(p + 1e-12))
    au = -np.sum(p_star_pred * np.log(p_star_pred + 1e-12), axis=-1)

    # Epistemic Uncertainty (EU): mean(delta_p)
    eu = np.mean(delta_p_pred, axis=-1)

    # Total Uncertainty
    tu = au + eu

    results.append({
        "Model": model_name,
        "Mean_AU": np.mean(au),
        "Mean_EU": np.mean(eu),
        "Mean_TU": np.mean(tu),
        "Train_Time_sec": train_time_sec
    })

import pandas as pd
results_df = pd.DataFrame(results)
summary_path = MODEL_DIR / "credit_uncertainty_summary.csv"
results_df.to_csv(summary_path, index=False)
print("Saved CREDIT uncertainty summary to", summary_path)
print(results_df)





========================= Distilling CREDIT Student for AlexNet_CNN =========================
Generating targets for training set...
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_AlexNet_CNN/AlexNet_CNN_ens_1_final.keras
102/102 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_AlexNet_CNN/AlexNet_CNN_ens_3_final.keras
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_AlexNet_CNN/AlexNet_CNN_ens_5_final.keras
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_AlexNet_CNN/AlexNet_CNN_ens_4_final.keras
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_AlexNet_CNN/AlexNet_CNN_ens_2_final.keras
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step
Generating targets for test set...
Loading teacher model: /content/drive/My 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'gf_patch_encoder', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


81/81 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_GFNet/GFNet_ens_3_final.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_GFNet/GFNet_ens_4_final.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_GFNet/GFNet_ens_1_final.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_GFNet/GFNet_ens_2_final.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step
Generating targets for test set...
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_GFNet/GFNet_ens_5_final.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 109ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_models/ensembles_GFNet/GFNet_ens_3_final.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 72ms/step
Loading teacher model: /content/drive/My Drive/m_p/saved_mod

81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 301ms/step - delta_p_loss: 0.0464 - loss: 1.6836 - p_star_loss: 1.6604
Epoch 1: val_loss improved from None to 0.97804, saving model to /content/drive/My Drive/m_p/saved_models/GFNet_CREDIT_best.keras

Epoch 1: finished saving model to /content/drive/My Drive/m_p/saved_models/GFNet_CREDIT_best.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 56s 372ms/step - delta_p_loss: 0.0166 - loss: 1.2302 - p_star_loss: 1.2210 - val_delta_p_loss: 0.0021 - val_loss: 0.9780 - val_p_star_loss: 0.9777
Epoch 2/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - delta_p_loss: 0.0041 - loss: 0.7518 - p_star_loss: 0.7498
Epoch 2: val_loss improved from 0.97804 to 0.52253, saving model to /content/drive/My Drive/m_p/saved_models/GFNet_CREDIT_best.keras

Epoch 2: finished saving model to /content/drive/My Drive/m_p/saved_models/GFNet_CREDIT_best.keras
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 103ms/step - delta_p_loss: 0.0034 - loss: 0.7008 - p_star_loss: 0.6988 - val_delta_p_loss: 0.0017 - val_loss: 0.5225 - v

In [18]:
# ==============================================================================
# CREDIT Inference & Evaluation ONLY (No Training)
# ==============================================================================
import os
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix,
    classification_report, f1_score, log_loss
)

sns.set_style("whitegrid")

# --- CONFIGURATION ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

MODEL_DIR = MODELS_DIR
DATA_DIR = REPO_ROOT / "data"
DATA_FILE = DATA_DIR / "pavia.mat"
LABEL_FILE = DATA_DIR / "pavia_gt.mat"

PATCH_SIZE = 9
INNER_PATCH = 3
TRAIN_PERCENT = 0.75
VAL_SPLIT_FROM_TRAIN = 0.20
BATCH_SIZE = 128

ALEXNET_CFG = {"conv_filters": [96, 256, 384, 384, 256], "dense_units": [4096, 1024, 256, 32]}
GFNET_CFG = {"hidden_dim": 512, "num_blocks": 5, "mlp_ratio": 4}
VIT_CFG = {"projection_dim": 256, "num_heads": 4, "transformer_layers": 12, "mlp_multiplier": 2, "head_units": [512, 256, 128, 64]}

ALEXNET_LEGACY_SPLIT_SEED = 10
ALEXNET_LEGACY_TRAIN_PERCENT = 0.75

# --- DATA LOADING ---
print("Loading Data...")
def extract_labeled_patches(x, y, patch_size=9):
    pad = patch_size // 2
    x_pad = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), mode="edge")
    coords = np.argwhere(y > 0)
    patches = np.empty((coords.shape[0], patch_size, patch_size, x.shape[-1]), dtype=np.float32)
    labels = np.empty((coords.shape[0],), dtype=np.int32)
    for i, (r, c) in enumerate(coords):
        patches[i] = x_pad[r:r + patch_size, c:c + patch_size, :]
        labels[i] = int(y[r, c]) - 1
    return patches, labels, coords


# -----------------------------
# Generalized Data Loading
# -----------------------------
import scipy.io as sio
import pandas as pd
import numpy as np

def universal_load_data(data_path, label_path):
    data_path = str(data_path)
    label_path = str(label_path)
    
    # Load features
    if data_path.endswith('.mat'):
        mat = sio.loadmat(data_path)
        x = next(v for k, v in mat.items() if not k.startswith('__') and isinstance(v, np.ndarray))
    elif data_path.endswith('.csv'):
        x = pd.read_csv(data_path).to_numpy(dtype=np.float32)
    elif data_path.endswith(('.tif', '.tiff')):
        try:
            import rasterio
            with rasterio.open(data_path) as src:
                x = src.read()
                x = np.moveaxis(x, 0, -1)
        except ImportError:
            print("rasterio not installed. Using dummy data.")
            x = np.zeros((10,10,3))

    # Load labels
    if label_path.endswith('.mat'):
        lmat = sio.loadmat(label_path)
        y = next(v for k, v in lmat.items() if not k.startswith('__') and isinstance(v, np.ndarray))
    elif label_path.endswith('.csv'):
        y = pd.read_csv(label_path).to_numpy(dtype=np.int32)
    elif label_path.endswith(('.tif', '.tiff')):
        try:
            import rasterio
            with rasterio.open(label_path) as src:
                y = src.read(1)
        except ImportError:
            y = np.zeros((10,10))

    # Normalization for 3D tensors
    if len(x.shape) == 3:
        x_norm = np.empty_like(x, dtype=np.float32)
        for b_idx in range(x.shape[-1]):
            band = x[:, :, b_idx]
            b_min, b_max = np.min(band), np.max(band)
            x_norm[:, :, b_idx] = (band - b_min) / max(b_max - b_min, 1e-8)
        x = x_norm
        
    return x, y

# Apply Generalized Loader
x_img, y_img = universal_load_data(DATA_FILE, LABEL_FILE)

# Handle flat CSVs by requesting user input or fallback
if len(x_img.shape) == 3:
    H, W, B = x_img.shape
else:
    print("WARNING: Data is flat. Please manually reshape x_img and y_img, then define H, W, B.")
    H, W, B = 330, 307, 6 # Default fallback for flat data

num_classes = int(np.unique(y_img).size)
print(f"Loaded Data Shape: {x_img.shape}, Labels Shape: {y_img.shape}, Classes: {num_classes}")

# Dynamic Color Palette Setup
import seaborn as sns
from matplotlib.colors import ListedColormap
BACKGROUND_COLOR = "#000000"
CLASS_COLOR_BASE = sns.color_palette("hls", max(10, num_classes)).as_hex()
X, y, coords = extract_labeled_patches(x_img, y_img, PATCH_SIZE)
num_classes = int(np.unique(y).size)
input_shape = (PATCH_SIZE, PATCH_SIZE, B)

# Standard Split
x_train_full, x_test, y_train_full, y_test = train_test_split(X, y, train_size=TRAIN_PERCENT, random_state=SEED, stratify=y)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

# AlexNet Split
x_train_alex, x_test_alex, y_train_alex, y_test_alex = train_test_split(X, y, train_size=ALEXNET_LEGACY_TRAIN_PERCENT, random_state=ALEXNET_LEGACY_SPLIT_SEED, stratify=y)
y_test_alex_cat = keras.utils.to_categorical(y_test_alex, num_classes)

# --- METRIC HELPERS ---
def multiclass_brier_score(y_onehot, y_prob):
    return float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))

def expected_calibration_error(y_true, y_prob, n_bins=15):
    confidences, predictions = np.max(y_prob, axis=1), np.argmax(y_prob, axis=1)
    correct = (predictions == y_true).astype(np.float32)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        in_bin = (confidences >= bin_edges[i]) & (confidences <= bin_edges[i+1] if i == n_bins - 1 else confidences < bin_edges[i+1])
        prop = np.mean(in_bin)
        if prop > 0:
            ece += np.abs(np.mean(correct[in_bin]) - np.mean(confidences[in_bin])) * prop
    return float(ece)

# --- BASE ARCHITECTURE DEFINITIONS (Required to build the graph before loading weights) ---
# (Using placeholders here for brevity, KEEP YOUR EXISTING build_alexnet, build_gfnet, build_vit_unet here!)
# Just copy-paste the `build_alexnet`, `build_gfnet`, `build_vit_unet`, and custom Keras classes from the previous script here.
# I am assuming they are defined in memory.

def build_credit_student(base_builder_func, num_classes):
    base_model = base_builder_func()
    features = base_model.layers[-2].output
    p_star = tf.keras.layers.Dense(num_classes, activation='softmax', name='p_star')(features)
    delta_p = tf.keras.layers.Dense(num_classes, activation='sigmoid', name='delta_p')(features)
    return tf.keras.Model(inputs=base_model.input, outputs=[p_star, delta_p], name="CREDIT_Student")

model_builders = {
    "AlexNet_CNN": lambda: build_alexnet(input_shape, num_classes, dropout_rate=0.25, cfg=ALEXNET_CFG),
    "GFNet": lambda: build_gfnet(input_shape, num_classes, inner_patch=INNER_PATCH, hidden_dim=GFNET_CFG["hidden_dim"], num_blocks=GFNET_CFG["num_blocks"], mlp_ratio=GFNET_CFG["mlp_ratio"], dropout_rate=0.25),
    "ViT_UNet": lambda: build_vit_unet_singlehead(input_shape, num_classes, inner_patch=INNER_PATCH, projection_dim=VIT_CFG["projection_dim"], num_heads=VIT_CFG["num_heads"], transformer_layers=VIT_CFG["transformer_layers"], mlp_multiplier=VIT_CFG["mlp_multiplier"], dropout_rate=0.25, head_units=tuple(VIT_CFG["head_units"])),
}

# --- EVALUATION LOOP (NO TRAINING) ---
results = []
credit_artifacts = {}

for model_name, builder in model_builders.items():
    best_path = MODEL_DIR / f"{model_name}_CREDIT_best.keras"

    if not best_path.exists():
        print(f"Skipping {model_name}... No saved CREDIT weights found at {best_path}")
        continue

    print(f"\n{'=' * 25} Evaluating Saved CREDIT Model: {model_name} {'=' * 25}")
    tf.keras.backend.clear_session()

    # 1. Rebuild pristine graph and load weights
    student_model = build_credit_student(builder, num_classes)
    student_model.load_weights(best_path)

    # 2. Select correct test set
    if model_name == "AlexNet_CNN":
        x_te, y_te_true, y_te_cat = x_test_alex, y_test_alex, y_test_alex_cat
    else:
        x_te, y_te_true, y_te_cat = x_test, y_test, y_test_cat

    # 3. Predict & Extract Vectors
    print(f"Running test set predictions...")
    p_star_pred, delta_p_pred = student_model.predict(x_te, batch_size=BATCH_SIZE)

    au = -np.sum(p_star_pred * np.log(p_star_pred + 1e-12), axis=-1)
    eu = np.mean(delta_p_pred, axis=-1)
    tu = au + eu

    y_test_pred = np.argmax(p_star_pred, axis=-1)

    # 4. Generate Classification Metrics
    credit_artifacts[model_name] = {
        "confusion_matrix": confusion_matrix(y_te_true, y_test_pred),
        "report": classification_report(y_te_true, y_test_pred, output_dict=True, zero_division=0)
    }

    results.append({
        "Model": model_name,
        "Test_Accuracy": float(accuracy_score(y_te_true, y_test_pred)),
        "Macro_F1": float(f1_score(y_te_true, y_test_pred, average="macro")),
        "Cohen_Kappa": float(cohen_kappa_score(y_te_true, y_test_pred)),
        "Test_NLL": float(log_loss(y_te_true, p_star_pred, labels=np.arange(num_classes))),
        "Test_Brier": float(multiclass_brier_score(y_te_cat, p_star_pred)),
        "Test_ECE": float(expected_calibration_error(y_te_true, p_star_pred, n_bins=15)),
        "Mean_AU": np.mean(au),
        "Mean_EU": np.mean(eu),
        "Mean_TU": np.mean(tu)
    })

# --- CLASSIFICATION PLOTS ---
results_df = pd.DataFrame(results)
print("\n--- CREDIT Evaluation Summary ---")
print(results_df)

if len(credit_artifacts) > 0:
    class_ticks = [str(i + 1) for i in range(num_classes)]
    fig, axes = plt.subplots(1, len(credit_artifacts), figsize=(7 * len(credit_artifacts), 5.5))
    if len(credit_artifacts) == 1: axes = [axes]

    for ax, (model_name, artifact) in zip(axes, credit_artifacts.items()):
        sns.heatmap(artifact["confusion_matrix"], annot=True, fmt="d", cmap="Blues", xticklabels=class_ticks, yticklabels=class_ticks, cbar=False, ax=ax)
        ax.set_title(f"{model_name} CREDIT\nConfusion Matrix")
        ax.set_xlabel("Predicted Class")
        ax.set_ylabel("True Class")
    plt.tight_layout()
    plt.savefig(MODEL_DIR / "credit_confusion_matrices.png", dpi=220, bbox_inches="tight")
    plt.show()

# --- 6-PANEL SPATIAL MAPPING (Apples-to-Apples Absolute Thresholding) ---
print("\nExtracting all spatial patches into flat array for full-scene inference...")
pad = PATCH_SIZE // 2
x_pad = np.pad(x_img, ((pad, pad), (pad, pad), (0, 0)), mode="edge")
scene_pixels_scaled = np.empty((H * W, PATCH_SIZE, PATCH_SIZE, B), dtype=np.float32)
idx = 0
for r in range(H):
    for c in range(W):
        scene_pixels_scaled[idx] = x_pad[r:r + PATCH_SIZE, c:c + PATCH_SIZE, :]
        idx += 1

def generate_spatial_credit_masks_standardized(model_name, student_model, scene_pixels, H, W, au_thresh=0.5, eu_thresh=0.2, tu_thresh=0.7):
    print(f"\nGenerating 6-Panel Spatial Maps for {model_name}...")
    p_star_scene, delta_p_scene = student_model.predict(scene_pixels, batch_size=2048, verbose=1)
    p_star_scene = np.clip(p_star_scene, 1e-12, 1.0)

    n_classes = p_star_scene.shape[-1]
    au_scene = -np.sum(p_star_scene * np.log(p_star_scene), axis=-1)
    eu_scene = np.mean(delta_p_scene, axis=-1)
    tu_scene = au_scene + eu_scene
    pred_class_map = np.argmax(p_star_scene, axis=-1).reshape((H, W))

    au_mask = (au_scene.reshape((H, W)) > au_thresh).astype(int)
    eu_mask = (eu_scene.reshape((H, W)) > eu_thresh).astype(int)
    tu_mask = (tu_scene.reshape((H, W)) > tu_thresh).astype(int)

    combined_au = np.where(au_mask == 1, n_classes, pred_class_map)
    combined_eu = np.where(eu_mask == 1, n_classes, pred_class_map)
    combined_tu = np.where(tu_mask == 1, n_classes, pred_class_map)

#     CLASS_COLOR_BASE = ['#0000FF', '#00FF00', '#FF0000', '#00FFFF', '#FF00FF', '#FFFF00', '#A52A2A', '#FFA500', '#7FFF00', '#8A2BE2']
    cmap_classes = ListedColormap(CLASS_COLOR_BASE[:n_classes])
    cmap_with_unc = ListedColormap(CLASS_COLOR_BASE[:n_classes] + ['#808080'])
    cmap_binary = ListedColormap(['#FFFF00', '#001F3F'])

    fig, axes = plt.subplots(2, 3, figsize=(28, 16))
    fig.suptitle(f"{model_name} CREDIT Uncertainty (Absolute Cutoffs)", fontsize=22, fontweight='bold')

    # Base Prediction
    axes[0, 0].imshow(pred_class_map, cmap=cmap_classes, vmin=0, vmax=n_classes-1)
    axes[0, 0].set_title("Base Prediction Map (No Uncertainty)", fontsize=16)
    axes[0, 0].axis('off')

    # Binary TU
    axes[0, 1].imshow(tu_mask, cmap=cmap_binary, vmin=0, vmax=1)
    axes[0, 1].set_title(f"Certain vs Uncertain Map (Total Unc. > {tu_thresh})", fontsize=16)
    axes[0, 1].axis('off')
    axes[0, 1].legend(handles=[Patch(facecolor='#FFFF00', edgecolor='black', label='Certain'), Patch(facecolor='#001F3F', edgecolor='black', label='Uncertain')], loc='center left', bbox_to_anchor=(1.02, 0.5), framealpha=1.0, fontsize=14)

    # Bar Chart
    uniq, cnt = np.unique(combined_tu, return_counts=True)
    c_dict = {int(k): int(v) for k, v in zip(uniq, cnt)}
    vals = [c_dict.get(i, 0) for i in range(n_classes + 1)]
    axes[0, 2].bar([f"Class {i}" for i in range(n_classes)] + ["Uncertain"], vals, color=CLASS_COLOR_BASE[:n_classes] + ['#808080'], edgecolor='black')
    axes[0, 2].set_title(f"Pixel Counts per Class (TU > {tu_thresh})", fontsize=16)
    axes[0, 2].tick_params(axis='x', rotation=45, labelsize=12)
    for i, v in enumerate(vals): axes[0, 2].text(i, v + (max(vals) if vals else 1)*0.01, f'{v:,}', ha='center', va='bottom', fontweight='bold')

    # Grey Overlays
    axes[1, 0].imshow(combined_au, cmap=cmap_with_unc, vmin=0, vmax=n_classes)
    axes[1, 0].set_title(f"Aleatoric Mask Overlay (Grey AU > {au_thresh})", fontsize=16)
    axes[1, 0].axis('off')

    axes[1, 1].imshow(combined_eu, cmap=cmap_with_unc, vmin=0, vmax=n_classes)
    axes[1, 1].set_title(f"Epistemic Mask Overlay (Grey EU > {eu_thresh})", fontsize=16)
    axes[1, 1].axis('off')

    axes[1, 2].imshow(combined_tu, cmap=cmap_with_unc, vmin=0, vmax=n_classes)
    axes[1, 2].set_title(f"Total Mask Overlay (Grey TU > {tu_thresh})", fontsize=16)
    axes[1, 2].axis('off')

    plt.tight_layout(rect=[0, 0, 0.95, 1])
    plt.savefig(MODEL_DIR / f"{model_name}_CREDIT_Spatial_Standardized.png", dpi=300, bbox_inches='tight')
    plt.show()

# Final Spatial Generation
for model_name, builder in model_builders.items():
    best_path = MODEL_DIR / f"{model_name}_CREDIT_best.keras"
    if best_path.exists():
        tf.keras.backend.clear_session()
        student_model = build_credit_student(builder, num_classes)
        student_model.load_weights(best_path)
        generate_spatial_credit_masks_standardized(model_name, student_model, scene_pixels_scaled, H, W)


Output hidden; open in https://colab.research.google.com to view.